In [ ]:
import sys
from pathlib import Path

def is_google_colab() -> bool:
    if "google.colab" in str(get_ipython()):
        return True
    return False

def clone_repository() -> None:
    !git clone https://github.com/featurestorebook/mlfs-book.git
    %cd mlfs-book

def install_dependencies() -> None:
    !pip install --upgrade uv
    !uv pip install --all-extras --system --requirement pyproject.toml

if is_google_colab():
    clone_repository()
    install_dependencies()
    root_dir = str(Path().absolute())
    print("Google Colab environment")
else:
    root_dir = Path().absolute()
    if root_dir.parts[-1:] == ('pollen',):
        root_dir = Path(*root_dir.parts[:-1])
    if root_dir.parts[-1:] == ('notebooks',):
        root_dir = Path(*root_dir.parts[:-1])
    root_dir = str(root_dir) 
    print("Local environment")

# Add the root directory to the `PYTHONPATH` to use the `recsys` Python module from the notebook.
if root_dir not in sys.path:
    sys.path.append(root_dir)
print(f"Added the following directory to the PYTHONPATH: {root_dir}")
    
# Set the environment variables from the file <root_dir>/.env
from mlfs import config
settings = config.HopsworksSettings(_env_file=f"{root_dir}/.env")

# Daily Feature Pipeline for Grass Pollen (Stockholm)

## Sections:
1. Fetch Pollen Data from Pollenrapporten API
2. Insert into Feature Group

**Schedule this notebook to run daily during pollen season (May-August)**

In [ ]:
import datetime
import pandas as pd
import hopsworks
from mlfs.airquality import util
import warnings
import json
warnings.filterwarnings("ignore")

## Connect to Hopsworks

In [ ]:
project = hopsworks.login(engine="python")
fs = project.get_feature_store()
secrets = hopsworks.get_secrets_api()

location_str = secrets.get_secret("SENSOR_LOCATION_JSON").value
location = json.loads(location_str)
country=location['country']
city=location['city']
street=location['street']

# Stockholm coordinates
latitude = location['latitude']
longitude = location['longitude']

today = datetime.date.today()
print(f"Fetching pollen data for: {today}")

## Get Feature Group Reference

In [ ]:
grass_pollen_fg = fs.get_feature_group(
    name='grass_pollen',
    version=1
)

weather_fg = fs.get_feature_group(
    name='weather',
    version=1,
)

## Fetch Today's Pollen Data

In [ ]:
# Fetch last 7 days to ensure we get today's data
start_date = (today - datetime.timedelta(days=7)).strftime('%Y-%m-%d')
end_date = today.strftime('%Y-%m-%d')

pollen_df = util.get_historical_pollen(
    start_date=start_date,
    end_date=end_date
)

if not pollen_df.empty:
    # Get only today's data
    pollen_df['date'] = pd.to_datetime(pollen_df['date']).dt.date
    pollen_today = pollen_df[pollen_df['date'] == today]
    pollen_today['date'] = pd.to_datetime(pollen_today['date'])
    print(f"Pollen data retrieved: {len(pollen_today)} records")
    print(pollen_today)
else:
    print("No pollen data available (likely outside monitoring season)")
    pollen_today = pd.DataFrame()

Get Weather Forecast data

In [ ]:
hourly_df = util.get_hourly_weather_forecast(city, latitude, longitude)
hourly_df = hourly_df.set_index('date')

# We will only make 1 daily prediction, so we will replace the hourly forecasts with a single daily forecast
# We only want the daily weather data, so only get weather at 12:00
daily_df = hourly_df.between_time('11:59', '12:01')
daily_df = daily_df.reset_index()
daily_df['date'] = pd.to_datetime(daily_df['date']).dt.date
daily_df['date'] = pd.to_datetime(daily_df['date'])
daily_df['city'] = city
daily_df

## Upload to Feature Store

In [ ]:
if not pollen_today.empty:
    grass_pollen_fg.insert(pollen_today, wait=True)
    print("✅ Pollen data uploaded successfully")
else:
    print("⚠️ No data to upload")

In [ ]:
# Insert new data
weather_fg.insert(daily_df, wait=True)